In [1]:
import os
import sys
import time
import json
from pathlib import Path
import torch
import pandas as pd

print("=" * 65)
print("ISLiM END-TO-END INFERENCE (TARGETED DOWNLOAD & RUN)")
print("=" * 65)

# --- 1. Clone Repo for Code & Splits (if missing) ---
REPO_DIR = Path("/kaggle/working/ISLiM")
if not REPO_DIR.exists():
    print("[+] Cloning ISLiM repository...")
    !git clone https://github.com/ahmedshoaib/ISLiM.git /kaggle/working/ISLiM

!pip install -q transformers sentencepiece gdown

# --- 2. Direct File Download (ONLY the 9 required files) ---
SLT_DIR = REPO_DIR / "checkpoints/mobile/android_test/app/build/intermediates/assets/debug/slt_models"
TEST_DIR = REPO_DIR / "checkpoints/mobile/android_test/app/build/intermediates/assets/debug/test_data"
SLT_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)

import gdown

# Exact file IDs extracted directly from the author's Google Drive folder
FILE_MAP = {
    # SLT Models
    SLT_DIR / "encoder_traced.ptl"        : "1xd6SCKhaGCY6K7OYpmwVR3rtRHX83tiy",
    SLT_DIR / "t5_encoder.ptl"            : "1QhRHsqzO_BkOZ8VLHL7XM0ViF3nMka0e",
    SLT_DIR / "t5_decoder_first_step.ptl" : "1d1_LADod4HfQnRx2OD_CsTzjbRo19xhN",
    SLT_DIR / "t5_decoder_step.ptl"       : "1Emx6mays0_5naLZJ-nm-zxIYP1uRisSn",
    SLT_DIR / "t5_lm_head.ptl"            : "17x3cQjzfrdX9mvVdJvG_6jrFqAF_qEyj",
    # Test Data
    TEST_DIR / "keypoints_module.pt"      : "1_llDyqX3Jgi32eweg3gPP8Hvol7Jq22d",
    TEST_DIR / "keypoints_module.ptl"     : "1dBvBxXMEVVDs9ReA-T2sUf83uWEWEP81",
    TEST_DIR / "references.txt"           : "180EF2F-w2VikjkkW_14TsYOL5ofAvDw2",
    TEST_DIR / "vocab.json"               : "1cKIZqyxLjnI3HByeLPVYhOnBKXV4ulC-",
}

print("[+] Checking and downloading the 9 model & test files directly...")
for dest_path, file_id in FILE_MAP.items():
    if not dest_path.exists() or dest_path.stat().st_size == 0:
        print(f"  Downloading {dest_path.name}...")
        gdown.download(id=file_id, output=str(dest_path), quiet=True)
    print(f"  [OK] {dest_path.name} ({dest_path.stat().st_size / 1e6:.2f} MB)")

# --- 3. Load Models & Tokenizer ---
print("\n[+] Loading model components...")
DEVICE = torch.device("cpu")

def _load(p):
    try: return torch.jit._load_for_lite_interpreter(str(p))
    except Exception: return torch.jit.load(str(p), map_location=DEVICE)

encoder = _load(SLT_DIR / "encoder_traced.ptl")
t5_encoder = _load(SLT_DIR / "t5_encoder.ptl")
decoder_first = _load(SLT_DIR / "t5_decoder_first_step.ptl")
decoder_step = _load(SLT_DIR / "t5_decoder_step.ptl")
lm_head = _load(SLT_DIR / "t5_lm_head.ptl")

from transformers import T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")
pad_token_id = 0
eos_token_id = 1
print("  All 5 components loaded successfully.")

# --- 4. Load References & Map to CSV ---
with open(TEST_DIR / "references.txt", "r", encoding="utf-8") as f:
    references = [l.strip().lower() for l in f if l.strip()]

CSV_PATH = REPO_DIR / "data/splits/isign_test_pose_stitch.csv"
df_test = pd.read_csv(CSV_PATH)
df_test["text_clean"] = df_test["text"].fillna("").astype(str).str.strip().str.lower()

aligned = []
for idx, ref in enumerate(references):
    m = df_test[df_test["text_clean"] == ref]
    assert len(m) == 1, f"Line {idx} did not have exactly 1 match in CSV."
    row = m.iloc[0]
    aligned.append({
        "sample_idx": idx,
        "csv_row_idx": m.index[0],
        "uid": str(row["uid"]),
        "ref_norm": ref,
        "ref_orig": str(row.get("text_original", ref))
    })

meta_df = pd.DataFrame(aligned)
print(f"[+] Aligned {len(meta_df)} ground-truth references to official UIDs.")

# --- 5. Extract Keypoints ---
mod_pt = torch.jit.load(str(TEST_DIR / "keypoints_module.pt"), map_location="cpu")
verified_keypoints = None

for name, tensor in dict(mod_pt.named_buffers()).items():
    if tensor.dim() == 4 and tensor.shape[1] == 4 and tensor.shape[2] == 94:
        verified_keypoints = tensor.detach().cpu().to(torch.float32)
        break

if verified_keypoints is None:
    schema = mod_pt.forward.schema
    if len(schema.arguments) == 1:
        verified_keypoints = mod_pt().detach().cpu().to(torch.float32)
    elif len(schema.arguments) == 2:
        arg_type = str(schema.arguments[1].type)
        samples = []
        for i in range(len(meta_df)):
            arg = i if "int" in arg_type else torch.tensor(i)
            s = mod_pt(arg)
            if s.dim() == 3: s = s.unsqueeze(0)
            samples.append(s.detach().cpu())
        verified_keypoints = torch.cat(samples, dim=0).to(torch.float32)

assert verified_keypoints is not None, "Failed to extract keypoints."
assert verified_keypoints.shape[0] == len(references), "Sample count mismatch."
print(f"[+] Keypoint Tensor: {verified_keypoints.shape}, dtype={verified_keypoints.dtype}")

# --- 6. Pure Greedy Generation Function ---
@torch.no_grad()
def translate(kp_tensor, max_len=128):
    t0 = time.time()
    kp = kp_tensor.to(device=DEVICE, dtype=torch.float32)
    if kp.dim() == 3: kp = kp.unsqueeze(0)

    vis = encoder(kp)
    mem = t5_encoder(vis)

    start = torch.full((1, 1), pad_token_id, dtype=torch.long, device=DEVICE)
    out1 = decoder_first(mem, start)
    hidden, sa_keys, sa_vals, ca_keys, ca_vals = out1

    logits = lm_head(hidden)
    next_tok = int(logits[0, -1].argmax(dim=-1).item())

    tokens = []
    if next_tok != eos_token_id:
        tokens.append(next_tok)

    for _ in range(1, max_len):
        if next_tok == eos_token_id:
            break
        t_in = torch.full((1, 1), next_tok, dtype=torch.long, device=DEVICE)
        out = decoder_step(mem, t_in, sa_keys, sa_vals, ca_keys, ca_vals)
        logits, sa_keys, sa_vals, ca_keys, ca_vals = out

        next_tok = int(logits[0, -1].argmax(dim=-1).item())
        if next_tok == eos_token_id:
            break
        tokens.append(next_tok)

    lat = time.time() - t0
    pred = tokenizer.decode(tokens, skip_special_tokens=True).strip().lower()
    return pred, tokens, lat

# --- 7. Translate Samples 0, 1, and 2 ---
print("\n" + "=" * 65)
print("ACTUAL TRANSLATIONS (SAMPLES 0, 1, AND 2)")
print("=" * 65)

for idx in [0, 1, 2]:
    meta = meta_df.iloc[idx]
    kp = verified_keypoints[idx:idx + 1]
    pred, tok_ids, lat = translate(kp)

    print(f"\n--- Sample {idx} ---")
    print(f"UID          : {meta['uid']}")
    print(f"Ground Truth : \"{meta['ref_norm']}\"")
    print(f"Original Text: \"{meta['ref_orig']}\"")
    print(f"Prediction   : \"{pred}\"")
    print(f"Token IDs    : {tok_ids}")
    print(f"Latency      : {lat * 1000:.1f} ms")

print("\n" + "=" * 65)
print("INFERENCE COMPLETE!")
print("=" * 65)

ISLiM END-TO-END INFERENCE (TARGETED DOWNLOAD & RUN)
[+] Cloning ISLiM repository...
Cloning into '/kaggle/working/ISLiM'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 65 (delta 10), reused 44 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 6.54 MiB | 3.77 MiB/s, done.
Resolving deltas: 100% (10/10), done.
[+] Checking and downloading the 9 model & test files directly...
  [OK] encoder_traced.ptl (57.92 MB)
  [OK] t5_encoder.ptl (75.56 MB)
  [OK] t5_decoder_first_step.ptl (166.57 MB)
  [OK] t5_decoder_step.ptl (153.98 MB)
  [OK] t5_lm_head.ptl (65.80 MB)
  [OK] keypoints_module.pt (10.98 MB)
  [OK] keypoints_module.ptl (10.98 MB)
  [OK] references.txt (0.00 MB)
  [OK] vocab.json (0.75 MB)

[+] Loading model components...


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

  All 5 components loaded successfully.
[+] Aligned 57 ground-truth references to official UIDs.
[+] Keypoint Tensor: torch.Size([57, 4, 94, 128]), dtype=torch.float32

ACTUAL TRANSLATIONS (SAMPLES 0, 1, AND 2)

--- Sample 0 ---
UID          : EO2U_VoYnbs--72
Ground Truth : "about how much more does the whale shark weigh"
Original Text: "about how much more does the whale shark weigh?"
Prediction   : "and the sprained roots"
Token IDs    : [11, 8, 3, 7, 102, 10761, 8523]
Latency      : 1539.4 ms

--- Sample 1 ---
UID          : kx6yqChkpxg--3
Ground Truth : "from ernakulam kerala a 1 0 year old girl named saanvi m prajit"
Original Text: "From Ernakulam, Kerala, a 10-year-old girl named Saanvi M Prajit"
Prediction   : "in kerala s kolkata a deaf girl named vijay singh"
Token IDs    : [16, 3, 2304, 138, 9, 3, 7, 3, 11292, 8682, 9, 3, 9, 20, 9, 89, 3202, 2650, 5931, 1191, 63, 10159, 107]
Latency      : 2860.2 ms

--- Sample 2 ---
UID          : UbSkIyCl70A--63
Ground Truth : "as they had 

In [2]:
!pip install -q sacrebleu jiwer

import sacrebleu
import jiwer
import numpy as np

print("=" * 65)
print("CELL 2: FULL TEST SET EVALUATION (57 SAMPLES)")
print("=" * 65)

all_predictions = []
all_references = meta_df["ref_norm"].tolist()
all_latencies = []

num_samples = len(meta_df)
print(f"[+] Translating all {num_samples} genuine test samples...")

for i in range(num_samples):
    sample_kp = verified_keypoints[i:i + 1]
    pred, tokens, lat = translate(sample_kp)
    all_predictions.append(pred)
    all_latencies.append(lat)
    
    # Progress log every 10 samples
    if (i + 1) % 10 == 0 or i == num_samples - 1:
        print(f"  [{i + 1:02d}/{num_samples}] Done (latency: {lat * 1000:.1f} ms)")

print("[+] All 57 samples translated successfully.")

# --- 1. Compute SacreBLEU (BLEU 1-4) & chrF++ ---
bleu_scores = {}
for n in range(1, 5):
    b = sacrebleu.metrics.BLEU(max_ngram_order=n, tokenize="13a")
    bleu_scores[f"BLEU-{n}"] = b.corpus_score(all_predictions, [all_references]).score

chrf_metric = sacrebleu.metrics.CHRF(word_order=2)
chrf_score = chrf_metric.corpus_score(all_predictions, [all_references]).score

# --- 2. Compute Word Error Rate (Corpus vs Average Sentence-Level) ---
# Corpus WER: Total edit distance divided by total reference words
corpus_wer = jiwer.wer(all_references, all_predictions) * 100

# Average Sentence-level WER: Mean of per-sentence WERs
sent_wers = [jiwer.wer(r, p) * 100 for r, p in zip(all_references, all_predictions) if len(r.strip()) > 0]
avg_sentence_wer = float(np.mean(sent_wers)) if sent_wers else 0.0

# --- 3. Summary Report ---
print("\n" + "=" * 65)
print("             ISLiM TEST EVALUATION SUMMARY")
print("=" * 65)
print(f"  Total Samples Evaluated : {num_samples}")
print(f"  Mean Latency (CPU)      : {np.mean(all_latencies) * 1000:.1f} ms / sample")
print("-" * 65)
for k, v in bleu_scores.items():
    print(f"  {k:<23} : {v:.2f}")
print(f"  chrF++                  : {chrf_score:.2f}")
print("-" * 65)
print(f"  Corpus WER              : {corpus_wer:.2f}%")
print(f"  Avg Sentence-level WER  : {avg_sentence_wer:.2f}%")
print("=" * 65)
print("  Paper Benchmark Context : Reported test BLEU-4 on iSign is 5.85")
print("=" * 65)

# --- 4. Display 8 Sample Predictions Side-by-Side ---
print("\nSample Comparisons (Predictions vs Ground Truth):")
for idx in [0, 1, 2, 5, 10, 15, 20, 25]:
    if idx < num_samples:
        print(f"\n[{idx:02d}] UID : {meta_df.iloc[idx]['uid']}")
        print(f"     REF : \"{all_references[idx]}\"")
        print(f"     PRED: \"{all_predictions[idx]}\"")

CELL 2: FULL TEST SET EVALUATION (57 SAMPLES)
[+] Translating all 57 genuine test samples...
  [10/57] Done (latency: 1721.6 ms)
  [20/57] Done (latency: 843.9 ms)
  [30/57] Done (latency: 2949.6 ms)
  [40/57] Done (latency: 3501.8 ms)
  [50/57] Done (latency: 2503.6 ms)
  [57/57] Done (latency: 2192.7 ms)
[+] All 57 samples translated successfully.

             ISLiM TEST EVALUATION SUMMARY
  Total Samples Evaluated : 57
  Mean Latency (CPU)      : 2176.1 ms / sample
-----------------------------------------------------------------
  BLEU-1                  : 24.20
  BLEU-2                  : 13.66
  BLEU-3                  : 9.00
  BLEU-4                  : 6.55
  chrF++                  : 20.49
-----------------------------------------------------------------
  Corpus WER              : 96.19%
  Avg Sentence-level WER  : 99.94%
  Paper Benchmark Context : Reported test BLEU-4 on iSign is 5.85

Sample Comparisons (Predictions vs Ground Truth):

[00] UID : EO2U_VoYnbs--72
     REF : 

In [3]:
print("=" * 65)
print("CELL 3: ISLiM TRANSLATION BACKEND WRAPPER")
print("=" * 65)

def translate_sign_keypoints(keypoints: torch.Tensor, max_length: int = 128) -> str:
    """
    Accepts preprocessed ISL keypoints from your CV pipeline and returns English text.
    
    Expected Input:
        keypoints: torch.Tensor of shape (1, 4, 94, 128) or (4, 94, 128)
                   Channels: [x, y, z, visibility]
                   Coordinates: Min-max normalized in [0, 1]
                   Frames: Resampled/padded to exactly 128 frames
    Returns:
        Translated English sentence string.
    """
    if keypoints.dim() == 3:
        keypoints = keypoints.unsqueeze(0)
    
    assert keypoints.shape == (1, 4, 94, 128), f"Expected shape (1, 4, 94, 128), got {keypoints.shape}"
    
    kp = keypoints.to(device=DEVICE, dtype=torch.float32)
    
    with torch.no_grad():
        # 1. Visual Encoder (Conv1D + Transformer) -> 16 query tokens
        vis = encoder(kp)
        
        # 2. T5 Encoder Memory
        mem = t5_encoder(vis)
        
        # 3. Autoregressive Greedy Generation
        start = torch.full((1, 1), pad_token_id, dtype=torch.long, device=DEVICE)
        out1 = decoder_first(mem, start)
        hidden, sa_keys, sa_vals, ca_keys, ca_vals = out1
        
        logits = lm_head(hidden)
        next_tok = int(logits[0, -1].argmax(dim=-1).item())
        
        tokens = []
        if next_tok != eos_token_id:
            tokens.append(next_tok)
            
        for _ in range(1, max_length):
            if next_tok == eos_token_id:
                break
            tok_in = torch.full((1, 1), next_tok, dtype=torch.long, device=DEVICE)
            out = decoder_step(mem, tok_in, sa_keys, sa_vals, ca_keys, ca_vals)
            logits, sa_keys, sa_vals, ca_keys, ca_vals = out
            
            next_tok = int(logits[0, -1].argmax(dim=-1).item())
            if next_tok == eos_token_id:
                break
            tokens.append(next_tok)
            
    return tokenizer.decode(tokens, skip_special_tokens=True).strip().lower()

# Quick test with Sample 0
test_pred = translate_sign_keypoints(verified_keypoints[0:1])
print(f"[+] Wrapper function ready! Test output: \"{test_pred}\"")

CELL 3: ISLiM TRANSLATION BACKEND WRAPPER
[+] Wrapper function ready! Test output: "and the sprained roots"


In [4]:

from pathlib import Path

REPO_DIR = Path("/kaggle/working/ISLiM")

# Search the source code for temporal processing
keywords = [
    "interpolate",
    "max_frames",
    "num_frames",
    "128",
    "keypoints",
    "pose_length",
    "resample",
    "temporal"
]

for file in REPO_DIR.rglob("*.py"):
    if any(part in str(file).lower() for part in ["site-packages", ".git"]):
        continue

    try:
        lines = file.read_text(encoding="utf-8").splitlines()
    except Exception:
        continue

    matches = [
        (i + 1, line.strip())
        for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    if matches:
        print(f"\n{'=' * 70}")
        print("FILE:", file.relative_to(REPO_DIR))
        print("=" * 70)

        for line_no, line in matches:
            print(f"{line_no}: {line}")



FILE: config.py
8: export ISIGN_KEYPOINTS_DIR=/path/to/keypoints
12: export GERMAN_KEYPOINTS_DIR=/path/to/phoenix/keypoints
33: ISIGN_KEYPOINTS_DIR = _env_dir(
34: "ISIGN_KEYPOINTS_DIR",
35: str(_PROJECT_ROOT / "data" / "keypoints"))
52: GERMAN_KEYPOINTS_DIR = _env_dir(
53: "GERMAN_KEYPOINTS_DIR", "~/Datasets/phoenix2014T/keypoints")
72: ("iSign keypoints", ISIGN_KEYPOINTS_DIR),

FILE: scripts/train.py
122: "keypoint_dim": 4, "num_keypoints": 94,
123: "temporal_channels": [376, 256, 256], "temporal_kernel": 5,
126: "output_dim": 512, "max_frames": 128,
131: "max_target_len": 128, "label_smoothing": args.label_smoothing,
183: kp = batch["keypoints"].to(device)
301: kp = batch["keypoints"].to(device)

FILE: data/splits/generate_pose_stitch_split.py
5: Split: 90/5/5, seed=42, UID-mapped to original keypoints.
19: KP_DIR = Path("/home/shoaib/phd/Datasets/iSign/misc/iSign_Keypoints")
96: # Filter rows with valid keypoints
107: print(f"  Valid rows with keypoints: {len(valid_rows)}")
108: p

In [12]:
import os
import numpy as np

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/elzarose/distillathon-data/train_augmented.jsonl


In [13]:
import numpy as np
import torch

input_path = "/kaggle/input/datasets/elzarose/how-are-you/islm_model_input.npy"

kp = torch.from_numpy(
    np.load(input_path)
).float()

print("Shape:", kp.shape)
print("Dtype:", kp.dtype)
print("NaN:", torch.isnan(kp).any().item())
print("Inf:", torch.isinf(kp).any().item())

Shape: torch.Size([1, 4, 94, 128])
Dtype: torch.float32
NaN: False
Inf: False


In [14]:
prediction = translate_sign_keypoints(kp)

print("\n========== WEBCAM SIGN TRANSLATION ==========")
print("Predicted sentence:", prediction)


========== WEBCAM SIGN TRANSLATION ==========
Predicted sentence: if you are interested in buying tickets you can contact them for the promotion


In [16]:
import numpy as np
import os

input_path = "/kaggle/input/datasets/elzarose/how-are-you/islm_model_input.npy"

print("File exists:", os.path.exists(input_path))
print("File size:", os.path.getsize(input_path), "bytes")

keypoints = np.load(input_path)

print("\nKaggle input:")
print("Shape:", keypoints.shape)
print("Mean:", keypoints.mean())
print("Std:", keypoints.std())
print("Min:", keypoints.min())
print("Max:", keypoints.max())

File exists: True
File size: 192640 bytes

Kaggle input:
Shape: (1, 4, 94, 128)
Mean: 0.6530594
Std: 0.33050197
Min: 0.0
Max: 1.0


In [17]:
print("Input checksum:", np.sum(keypoints))

Input checksum: 31430.445
